# Exp 007 Native Priors On Exp006 OOF

## Goal

Test whether the proven `exp_005` metadata priors and texture-aware smoothing become stronger when applied on top of the exported multi-fold validation predictions from `exp_006`.

## Hypothesis

`exp_006` is a mixed but still interesting native training branch. The most informative next step is not another raw training run, but checking whether the same soundscape-aware inference layer that helped `exp_004` also helps the stronger fold-exported `exp_006` branch.

## What Changes Here

- reuse exported `best_valid_outputs.npz` and `best_valid_meta.csv` from `exp_006`
- reconstruct the same grouped soundscape folds
- fit fold-safe metadata priors from the corresponding training partition
- compare raw predictions against:
  - event priors only
  - texture priors only
  - event + texture priors
  - event + texture priors + smoothing
- aggregate results across folds `0-2`

## Expected Outcome

A clear answer to whether the next native hybrid should be built on top of `exp_006` instead of the older `exp_004` outputs.


## Expected Readout

The notebook saves:

- `fold_results.csv`
- `oof_ablation_results.csv`
- `classwise_auc_comparison.csv`
- `result_snapshot.json`
- `best_oof_predictions.csv` (or parquet if available)

All outputs are written to `experiments/outputs/exp_007_native_priors_on_exp006_oof/`.


In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / 'data' / 'birdclef-2026').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve PROJECT_ROOT with data/birdclef-2026')


@dataclass
class EvalConfig:
    fold_ids: tuple[int, ...] = (0, 1, 2)
    n_folds: int = 5
    lambda_event: float = 0.4
    lambda_texture: float = 1.0
    smooth_texture: float = 0.35


CFG = EvalConfig()
DATA_DIR = PROJECT_ROOT / 'data' / 'birdclef-2026'
EXP006_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_006_soundscape_finetuning_v2'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_007_native_priors_on_exp006_oof'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    'project_root': str(PROJECT_ROOT),
    'exp006_dir': str(EXP006_DIR),
    'output_dir': str(OUTPUT_DIR),
    **asdict(CFG),
}


In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
TEXTURE_TAXA = {'Amphibia', 'Insecta'}


def parse_soundscape_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    return sorted(set(label for item in series for label in parse_soundscape_labels(item)))


def parse_soundscape_filename(name: str) -> dict:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


def encode_multi_hot(labels: list[str]) -> np.ndarray:
    target = np.zeros(len(PRIMARY_LABELS), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)
sc_clean = sc_clean.sort_values(['filename', 'end_sec']).reset_index(drop=True)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean['filename'].isin(full_files)].copy().reset_index(drop=True)

texture_labels = taxonomy.loc[taxonomy['class_name'].isin(TEXTURE_TAXA), 'primary_label'].tolist()
texture_set = set(texture_labels)
idx_texture = np.array([label_to_idx[label] for label in PRIMARY_LABELS if label in texture_set], dtype=np.int32)
idx_event = np.array([idx for idx, label in enumerate(PRIMARY_LABELS) if label not in texture_set], dtype=np.int32)

n_group_splits = min(CFG.n_folds, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df['filename']))

{
    'all_labeled_segments': int(len(sc_clean)),
    'fully_labeled_files': int(len(full_files)),
    'folds_available': [int(f) for f in CFG.fold_ids],
    'texture_classes': int(len(idx_texture)),
    'event_classes': int(len(idx_event)),
}


In [ ]:
def stable_logit(p: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float32), eps, 1.0 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def stable_sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    positive = x >= 0
    out = np.empty_like(x, dtype=np.float32)
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def score_macro_auc(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> dict:
    aucs = []
    skipped_no_positive = 0
    skipped_no_negative = 0
    for idx, _ in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0:
            skipped_no_positive += 1
            continue
        if negatives == 0:
            skipped_no_negative += 1
            continue
        aucs.append(float(roc_auc_score(y_true[:, idx], y_score[:, idx])))
    macro_auc = float(np.mean(aucs)) if aucs else float('nan')
    return {
        'macro_auc': macro_auc,
        'scored_classes': len(aucs),
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray) -> dict:
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    for site in site_keys:
        idx = site_to_i[site]
        mask = prior_df['site'].astype(str).values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = prior_df['hour_utc'].astype(int).values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return stable_logit(p, eps=eps)


def smooth_cols_by_file(scores: np.ndarray, meta_df: pd.DataFrame, cols: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    smoothed = scores.copy()
    ordered_meta = meta_df[['filename', 'end_sec']].reset_index(drop=True)
    for _, group_idx in ordered_meta.groupby('filename', sort=False).groups.items():
        group_idx = np.array(list(group_idx))
        if len(group_idx) <= 1:
            continue
        x = smoothed[group_idx][:, cols]
        prev_x = np.concatenate([x[:1], x[:-1]], axis=0)
        next_x = np.concatenate([x[1:], x[-1:]], axis=0)
        smoothed[np.ix_(group_idx, cols)] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return smoothed


def fuse_native_logits(logits: np.ndarray, prior_logits: np.ndarray, meta_df: pd.DataFrame, lambda_event: float, lambda_texture: float, smooth_texture: float = 0.0) -> np.ndarray:
    fused = logits.copy()
    if len(idx_event):
        fused[:, idx_event] += lambda_event * prior_logits[:, idx_event]
    if len(idx_texture):
        fused[:, idx_texture] += lambda_texture * prior_logits[:, idx_texture]
    if smooth_texture > 0:
        fused = smooth_cols_by_file(fused, meta_df, idx_texture, alpha=smooth_texture)
    return fused


def classwise_auc_table(y_true: np.ndarray, y_score: np.ndarray, labels: list[str]) -> pd.DataFrame:
    rows = []
    for idx, label in enumerate(labels):
        positives = int(y_true[:, idx].sum())
        negatives = int(len(y_true) - positives)
        if positives == 0 or negatives == 0:
            auc = np.nan
        else:
            auc = float(roc_auc_score(y_true[:, idx], y_score[:, idx]))
        rows.append({
            'primary_label': label,
            'positives': positives,
            'negatives': negatives,
            'auc': auc,
        })
    return pd.DataFrame(rows)


In [ ]:
fold_rows = []
fold_prediction_exports = []
oof_variant_parts = {}
variant_settings = [
    ('raw', None),
    ('event_priors_only', (CFG.lambda_event, 0.0, 0.0)),
    ('texture_priors_only', (0.0, CFG.lambda_texture, 0.0)),
    ('event_texture_priors', (CFG.lambda_event, CFG.lambda_texture, 0.0)),
    ('event_texture_priors_smooth', (CFG.lambda_event, CFG.lambda_texture, CFG.smooth_texture)),
]

for fold in CFG.fold_ids:
    train_idx, valid_idx = full_splits[fold]
    valid_files = set(full_df.iloc[valid_idx]['filename'].tolist())

    train_df = (
        sc_clean[~sc_clean['filename'].isin(valid_files)]
        .sort_values(['filename', 'end_sec'])
        .reset_index(drop=True)
    )
    valid_df = (
        full_df[full_df['filename'].isin(valid_files)]
        .sort_values(['filename', 'end_sec'])
        .reset_index(drop=True)
    )

    fold_dir = EXP006_DIR / f'fold_{fold:02d}'
    snapshot = json.loads((fold_dir / 'result_snapshot.json').read_text())
    meta_df = pd.read_csv(fold_dir / 'best_valid_meta.csv').sort_values(['filename', 'end_sec']).reset_index(drop=True)
    outputs = np.load(fold_dir / 'best_valid_outputs.npz')
    y_true = outputs['y_true'].astype(np.float32)
    raw_probs = outputs['y_pred'].astype(np.float32)
    raw_logits = stable_logit(raw_probs)

    if len(meta_df) != len(valid_df):
        raise ValueError(f'Fold {fold}: exported meta rows do not match reconstructed valid rows: {len(meta_df)} vs {len(valid_df)}')
    if meta_df['row_id'].tolist() != valid_df['row_id'].tolist():
        raise ValueError(f'Fold {fold}: row_id mismatch between exported meta and reconstructed validation fold')

    y_train = np.stack([encode_multi_hot(labels) for labels in train_df['label_list']], axis=0)
    prior_tables = fit_prior_tables(train_df[['site', 'hour_utc']], y_train)
    prior_logits = prior_logits_from_tables(
        sites=meta_df['site'].to_numpy(),
        hours=meta_df['hour_utc'].to_numpy(),
        tables=prior_tables,
    )

    fold_variant_scores = {}
    for name, settings in variant_settings:
        if settings is None:
            probs = raw_probs.copy()
        else:
            lambda_event, lambda_texture, smooth_texture = settings
            fused_logits = fuse_native_logits(
                logits=raw_logits,
                prior_logits=prior_logits,
                meta_df=meta_df,
                lambda_event=lambda_event,
                lambda_texture=lambda_texture,
                smooth_texture=smooth_texture,
            )
            probs = stable_sigmoid(fused_logits)
        fold_variant_scores[name] = probs
        metrics = score_macro_auc(y_true, probs, PRIMARY_LABELS)
        fold_rows.append({
            'fold': int(fold),
            'variant': name,
            **metrics,
        })
        oof_variant_parts.setdefault(name, []).append((fold, meta_df.copy(), y_true.copy(), probs.copy()))

    fold_ablation = pd.DataFrame([row for row in fold_rows if row['fold'] == fold]).sort_values('macro_auc', ascending=False).reset_index(drop=True)
    best_variant = fold_ablation.iloc[0]['variant']
    best_scores = fold_variant_scores[best_variant]
    fold_export_base = meta_df.copy()
    fold_export_base['fold'] = int(fold)
    fold_export_base['best_variant'] = best_variant
    raw_export = pd.DataFrame(raw_probs.astype(np.float32), columns=[f'raw__{label}' for label in PRIMARY_LABELS])
    best_export = pd.DataFrame(best_scores.astype(np.float32), columns=[f'{best_variant}__{label}' for label in PRIMARY_LABELS])
    fold_prediction_exports.append(pd.concat([fold_export_base.reset_index(drop=True), raw_export, best_export], axis=1))

fold_results = pd.DataFrame(fold_rows).sort_values(['fold', 'macro_auc'], ascending=[True, False]).reset_index(drop=True)
fold_results.to_csv(OUTPUT_DIR / 'fold_results.csv', index=False)
print(fold_results)


In [ ]:
oof_rows = []
variant_predictions_for_export = {}
variant_targets = None
variant_meta = None

for name, parts in oof_variant_parts.items():
    parts = sorted(parts, key=lambda item: item[0])
    meta_concat = pd.concat([item[1] for item in parts], axis=0).reset_index(drop=True)
    y_true_concat = np.concatenate([item[2] for item in parts], axis=0)
    y_pred_concat = np.concatenate([item[3] for item in parts], axis=0)
    metrics = score_macro_auc(y_true_concat, y_pred_concat, PRIMARY_LABELS)
    oof_rows.append({'variant': name, **metrics})
    variant_predictions_for_export[name] = y_pred_concat
    if variant_targets is None:
        variant_targets = y_true_concat
        variant_meta = meta_concat


oof_ablation = pd.DataFrame(oof_rows).sort_values('macro_auc', ascending=False).reset_index(drop=True)
oof_ablation.to_csv(OUTPUT_DIR / 'oof_ablation_results.csv', index=False)

best_variant = oof_ablation.iloc[0]['variant']
raw_probs_oof = variant_predictions_for_export['raw']
best_probs_oof = variant_predictions_for_export[best_variant]

classwise_raw = classwise_auc_table(variant_targets, raw_probs_oof, PRIMARY_LABELS).rename(columns={'auc': 'raw_auc'})
classwise_best = classwise_auc_table(variant_targets, best_probs_oof, PRIMARY_LABELS).rename(columns={'auc': 'best_auc'})
classwise_compare = classwise_raw.merge(classwise_best[['primary_label', 'best_auc']], on='primary_label', how='left')
classwise_compare['delta'] = classwise_compare['best_auc'] - classwise_compare['raw_auc']
classwise_compare.to_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv', index=False)

best_export = pd.concat([
    variant_meta.reset_index(drop=True),
    pd.DataFrame({'best_variant': [best_variant] * len(variant_meta)}),
    pd.DataFrame(variant_targets.astype(np.int8), columns=[f'true__{label}' for label in PRIMARY_LABELS]),
    pd.DataFrame(raw_probs_oof.astype(np.float32), columns=[f'raw__{label}' for label in PRIMARY_LABELS]),
    pd.DataFrame(best_probs_oof.astype(np.float32), columns=[f'{best_variant}__{label}' for label in PRIMARY_LABELS]),
], axis=1)
try:
    best_export.to_parquet(OUTPUT_DIR / 'best_oof_predictions.parquet', index=False)
except Exception:
    best_export.to_csv(OUTPUT_DIR / 'best_oof_predictions.csv', index=False)

pd.concat(fold_prediction_exports, axis=0).to_csv(OUTPUT_DIR / 'fold_level_best_predictions.csv', index=False)

result_snapshot = {
    'experiment_id': 'exp_007',
    'experiment_name': 'native_priors_on_exp006_oof',
    'fold_ids': [int(f) for f in CFG.fold_ids],
    'raw_oof_macro_auc': float(oof_ablation.loc[oof_ablation['variant'] == 'raw', 'macro_auc'].iloc[0]),
    'best_variant': str(best_variant),
    'best_oof_macro_auc': float(oof_ablation.iloc[0]['macro_auc']),
    'delta_vs_raw': float(oof_ablation.iloc[0]['macro_auc'] - oof_ablation.loc[oof_ablation['variant'] == 'raw', 'macro_auc'].iloc[0]),
    'lambda_event': float(CFG.lambda_event),
    'lambda_texture': float(CFG.lambda_texture),
    'smooth_texture': float(CFG.smooth_texture),
    'oof_scored_classes_best_variant': int(oof_ablation.iloc[0]['scored_classes']),
    'fold_mean_raw_macro_auc': float(fold_results[fold_results['variant'] == 'raw']['macro_auc'].mean()),
    'fold_mean_best_macro_auc': float(fold_results.sort_values(['fold', 'macro_auc'], ascending=[True, False]).groupby('fold').head(1)['macro_auc'].mean()),
}
(OUTPUT_DIR / 'result_snapshot.json').write_text(json.dumps(result_snapshot, indent=2))

print(oof_ablation)
print('\nSnapshot:')
print(json.dumps(result_snapshot, indent=2))


In [ ]:
snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
print('Snapshot:')
print(json.dumps(snapshot, indent=2))

print('\nFold results:')
display(pd.read_csv(OUTPUT_DIR / 'fold_results.csv'))

print('\nOOF ablations:')
display(pd.read_csv(OUTPUT_DIR / 'oof_ablation_results.csv'))

compare_df = pd.read_csv(OUTPUT_DIR / 'classwise_auc_comparison.csv')
print('\nTop gains:')
display(compare_df.sort_values('delta', ascending=False).head(20))
print('\nTop regressions:')
display(compare_df.sort_values('delta', ascending=True).head(20))


## Reading The Result

The main question is not whether every fold improves equally.

The main question is whether `exp_006` becomes a meaningfully stronger native hybrid after applying the same soundscape-aware priors and texture logic that already helped `exp_004`.

Interpretation guide:

1. If the OOF best variant clearly beats raw `exp_006`, then `exp_006 + priors` becomes the right native hybrid base for the next Kaggle test.
2. If the uplift is very small or unstable, then the next real gain should likely come from a bigger modeling jump such as long-context SED or noisy-student pseudo-labeling.
3. If texture priors again dominate the uplift, that further confirms the competition's texture-heavy soundscape structure and justifies a future specialist branch.
